# Proyecto final: Sistema de Inferencia Difusa para Control de Lavavajillas

Autor: Esteban Suárez Calvo

In [73]:
import numpy as np
import pennylane as qml

## Especificación del problema

# Proyecto Final: Sistema de Inferencia Difusa para Control de Lavavajillas

El objetivo de este proyecto es desarrollar un modelo de inferencia difusa que determine el modo de lavado óptimo de un lavavajillas en base a la carga de platos y al nivel de grasa.

## Variables de entrada

Las variables de entrada son las siguientes:

1. **Carga de platos (0-100):** representa el porcentaje de ocupación del lavavajillas. Las etiquetas son las siguientes:
   1. *Poca*
   2. *Media*
   3. *Mucha*
2. **Nivel de grasa (0-100):** representa la cantidad de grasa que tienen los platos. Las etiquetas son las siguientes:
   1. *Ligera*
   2. *Normal*
   3. *Intensa*

## Variable de salida

La salida del sistema es una única variable, el **modo de lavado**, que tema las siguientes etiquetas:

1. *Eco* (30 min)
2. *Normal* (60 min)
3. *Intensivo* (90 min)

## Base de reglas

El sistema agrupa 9 reglas en las siguientes combinaciones mediante operadores AND y OR:

- SI (Poca Y Ligera) O (Poca Y Normal) O (Media Y Ligera) ENTONCES **Modo Eco**
- SI (Poca Y Intensa) O (Media Y Normal) O (Mucha Y Ligera) ENTONCES **Modo Normal**
- SI (Media Y Intensa) O (Mucha Y Normal) O (Mucha Y Intensa) ENTONCES **Modo Intensivo**

## Declaración del sistema clásico

Primero de todo, definimos las funciones de pertenencia base que utilizaremos más adelante.

In [74]:
def linzmf(x, a, b):
    if x <= a:
        return 1.0
    if x >= b:
        return 0.0
    return (b - x) / (b - a)


def linsmf(x, a, b):
    if x <= a:
        return 0.0
    if x >= b:
        return 1.0
    return (x - a) / (b - a)


def trimf(x, a, b, c):
    if x <= a or x >= c:
        return 0.0
    if a < x <= b:
        return (x - a) / (b - a)
    if b < x < c:
        return (c - x) / (c - b)
    return 0.0

Utilizando estas funciones de pertenencia definimos las variables difusas.

In [75]:
def fuzzy_carga(x):
    return {
        "poca": linzmf(x, 20, 50),
        "media": trimf(x, 20, 50, 80),
        "mucha": linsmf(x, 50, 80),
    }


def fuzzy_grasa(x):
    return {
        "ligera": linzmf(x, 20, 50),
        "normal": trimf(x, 20, 50, 80),
        "intensa": linsmf(x, 50, 80),
    }

Realizamos entonces la inferencia clásica tipo Sugeno. Destacar que para las reglas comobinadas utilizamos `min` para realizar las operaciones AND y `max` para realizar las operaciones OR.

In [76]:
def inferencia_clasica_lavavajillas(val_carga, val_grasa):
    carga = fuzzy_carga(val_carga)
    grasa = fuzzy_grasa(val_grasa)

    # Valores numéricos de la salida (minutos)
    out_eco = 30
    out_normal = 60
    out_intensivo = 90

    # 1. Evaluamos las reglas (Pesos de activación)
    w_eco = max(
        min(carga["poca"], grasa["ligera"]),
        min(carga["poca"], grasa["normal"]),
        min(carga["media"], grasa["ligera"]),
    )

    w_normal = max(
        min(carga["poca"], grasa["intensa"]),
        min(carga["media"], grasa["normal"]),
        min(carga["mucha"], grasa["ligera"]),
    )

    w_intensivo = max(
        min(carga["media"], grasa["intensa"]),
        min(carga["mucha"], grasa["normal"]),
        min(carga["mucha"], grasa["intensa"]),
    )

    pesos = [w_eco, w_normal, w_intensivo]

    # 2. OBTENER LA CATEGORÍA (El peso más alto gana)
    etiquetas = ["Eco", "Normal", "Intensivo"]
    indice_ganador = np.argmax(pesos)
    categoria_final = etiquetas[indice_ganador]

    # 3. OBTENER LOS MINUTOS EXACTOS (Defuzzificación por media ponderada)
    denominador = sum(pesos)
    minutos_exactos = (
        (w_eco * out_eco) + (w_normal * out_normal) + (w_intensivo * out_intensivo)
    ) / denominador

    return categoria_final, minutos_exactos, pesos

Por último probamos el sistema:

In [77]:
carga_test = 65
grasa_test = 40

categoria, minutos, pesos = inferencia_clasica_lavavajillas(carga_test, grasa_test)

print(f"Para una carga del {carga_test}% y una grasa del {grasa_test}%:")
print(f"Categoría recomendada: Modo {categoria}")
print(f"Tiempo exacto calculado: {minutos:.1f} minutos")
print(f"Pesos [Eco, Normal, Intensivo]={pesos}")

Para una carga del 65% y una grasa del 40%:
Categoría recomendada: Modo Normal
Tiempo exacto calculado: 63.8 minutos
Pesos [Eco, Normal, Intensivo]=[0.3333333333333333, 0.5, 0.5]


## Implementación del sistema cuántico

### Funciones auxiliares

A continuación definimos las funciones necesarias para el circuito. `encode_fuzzy_input` codifica los valores de pertenencia de carga y grasa en amplitudes de qubit. `init_quantum_fuzzy_output` prepara el registro de salida en superposición de las tres etiquetas. `apply_rule` aplica una puerta `MultiControlledX` para una combinación específica de entrada y etiqueta de salida. Por último, `defuzzify_quantum_output` convierte las probabilidades en un modo de lavado y un tiempo exacto.


In [78]:
def init_quantum_fuzzy_output(values, wires):
    # Inicializa |1> (x) (|00> + |01> + |10>) / sqrt(3) para 3 etiquetas
    n_labels = 3
    n_state = 2 ** len(wires)
    state = np.zeros(n_state, dtype=complex)

    # Índices para |1>|00>, |1>|01>, |1>|10> en 3 qubits son 4, 5, 6
    active_indices = [4, 5, 6]
    amp = 1 / np.sqrt(n_labels)
    for idx in active_indices:
        state[idx] = amp

    qml.AmplitudeEmbedding(state, wires=wires, normalize=True)


def encode_fuzzy_input(memberships, wires):
    # Codifica las pertenencias en amplitudes de 2 qubits: |00>, |01>, |10>
    amps = np.array(
        [
            np.sqrt(memberships[0]),
            np.sqrt(memberships[1]),
            np.sqrt(memberships[2]),
            0.0,
        ],
        dtype=float,
    )
    if np.allclose(amps, 0):
        amps[0] = 1.0  # Seguridad para evitar normas cero
    qml.AmplitudeEmbedding(amps, wires=wires, normalize=True)


def apply_rule(
    input_wires, input_values, output_label_bits, output_flag_wire, output_label_wires
):
    # Aplica la regla mediante un MultiControlledX
    control_wires = input_wires + output_label_wires
    control_values = input_values + output_label_bits
    qml.MultiControlledX(
        wires=control_wires + [output_flag_wire],
        control_values=control_values,
    )


def defuzzify_quantum_output(probs):
    # Usa solo los estados con flag=0: |000> (Eco), |001> (Normal), |010> (Intensivo)
    label_probs = probs[0:3]
    weights = np.array([30, 60, 90], dtype=float)  # Minutos: Eco, Normal, Intensivo
    denom = np.sum(label_probs)

    minutos_q = float(np.dot(label_probs, weights) / denom) if denom > 0 else 0.0

    etiquetas = ["Eco", "Normal", "Intensivo"]
    categoria_q = etiquetas[np.argmax(label_probs)]

    return categoria_q, minutos_q, label_probs

### Aplicación de reglas

En la parte clásica definimos 3 reglas agrupadas con AND y OR. Cada regla encapsula varias combinaciones de carga y grasa. En total son 9 combinaciones atómicas (3 × 3).

El circuito cuántico trabaja a nivel de estados base. Cada puerta `MultiControlledX` se activa solo para una combinación concreta. Por eso necesitamos **9 puertas, una por cada combinación atómica**.

Como cada combinación activa un único modo de salida, no hay conflictos. Las puertas que comparten el mismo modo actúan como un OR implícito.

A continuación definimos el circuito completo que implementa estas 9 reglas.


In [79]:
dev = qml.device("default.qubit", wires=7)


@qml.qnode(dev)
def fuzzy_inference_quantum_lavavajillas(val_carga, val_grasa):
    # Definición de cables
    carga_wires = [0, 1]
    grasa_wires = [2, 3]
    output_wires = [4, 5, 6]
    output_flag = output_wires[0]
    output_label = output_wires[1:]

    # 1. Fuzzificación Clásica
    c = fuzzy_carga(val_carga)
    g = fuzzy_grasa(val_grasa)

    # 2. Preparación del Estado (Inyección de Amplitudes)
    encode_fuzzy_input([c["poca"], c["media"], c["mucha"]], carga_wires)
    encode_fuzzy_input([g["ligera"], g["normal"], g["intensa"]], grasa_wires)

    # Inicializa la salida con la función de clase
    init_quantum_fuzzy_output(None, wires=output_wires)

    # 3. Reglas (Etiquetas en binario: 00=Eco, 01=Normal, 10=Intensivo)
    eco_bits = [0, 0]
    normal_bits = [0, 1]
    intensivo_bits = [1, 0]

    # --- MODO ECO ---
    # R1: Carga Poca (00) Y Grasa Ligera (00) -> Eco
    apply_rule(
        carga_wires + grasa_wires, [0, 0] + [0, 0], eco_bits, output_flag, output_label
    )
    # R2: Carga Poca (00) Y Grasa Normal (01) -> Eco
    apply_rule(
        carga_wires + grasa_wires, [0, 0] + [0, 1], eco_bits, output_flag, output_label
    )
    # R3: Carga Media (01) Y Grasa Ligera (00) -> Eco
    apply_rule(
        carga_wires + grasa_wires, [0, 1] + [0, 0], eco_bits, output_flag, output_label
    )

    # --- MODO NORMAL ---
    # R4: Carga Poca (00) Y Grasa Intensa (10) -> Normal
    apply_rule(
        carga_wires + grasa_wires,
        [0, 0] + [1, 0],
        normal_bits,
        output_flag,
        output_label,
    )
    # R5: Carga Media (01) Y Grasa Normal (01) -> Normal
    apply_rule(
        carga_wires + grasa_wires,
        [0, 1] + [0, 1],
        normal_bits,
        output_flag,
        output_label,
    )
    # R6: Carga Mucha (10) Y Grasa Ligera (00) -> Normal
    apply_rule(
        carga_wires + grasa_wires,
        [1, 0] + [0, 0],
        normal_bits,
        output_flag,
        output_label,
    )

    # --- MODO INTENSIVO ---
    # R7: Carga Media (01) Y Grasa Intensa (10) -> Intensivo
    apply_rule(
        carga_wires + grasa_wires,
        [0, 1] + [1, 0],
        intensivo_bits,
        output_flag,
        output_label,
    )
    # R8: Carga Mucha (10) Y Grasa Normal (01) -> Intensivo
    apply_rule(
        carga_wires + grasa_wires,
        [1, 0] + [0, 1],
        intensivo_bits,
        output_flag,
        output_label,
    )
    # R9: Carga Mucha (10) Y Grasa Intensa (10) -> Intensivo
    apply_rule(
        carga_wires + grasa_wires,
        [1, 0] + [1, 0],
        intensivo_bits,
        output_flag,
        output_label,
    )

    return qml.probs(wires=output_wires)

### Pruebas

In [80]:
carga_test = 65
grasa_test = 40

probs_q = fuzzy_inference_quantum_lavavajillas(carga_test, grasa_test)
cat_q, min_q, pesos_q = defuzzify_quantum_output(probs_q)

print(f"Categoría recomendada: Modo {cat_q}")
print(f"Tiempo exacto calculado: {min_q:.1f} minutos")
print(f"Probabilidades [Eco, Normal, Intensivo]: {np.round(pesos_q, 3)}")

Categoría recomendada: Modo Normal
Tiempo exacto calculado: 65.0 minutos
Probabilidades [Eco, Normal, Intensivo]: [0.056 0.167 0.111]


## Análisis de resultados

A continuación se comparan los resultados del sistema clásico (Sugeno) y el sistema cuántico para distintos valores de entrada. Se evalúan tanto la categoría recomendada como los minutos exactos calculados mediante defuzzificación.

In [81]:
casos = [
    (10, 10),
    (50, 20),
    (65, 40),
    (30, 70),
    (80, 80),
]

for carga_test, grasa_test in casos:
    # Sistema clásico
    cat_c, min_c, w_c = inferencia_clasica_lavavajillas(carga_test, grasa_test)

    # Sistema cuántico
    probs_q = fuzzy_inference_quantum_lavavajillas(carga_test, grasa_test)
    cat_q, min_q, w_q = defuzzify_quantum_output(probs_q)

    coinciden_cat = "SI" if cat_c == cat_q else "NO"

    print(f"Carga={carga_test}, Grasa={grasa_test}")
    print(
        f"  Clásico:  pesos=[{w_c[0]:.3f}, {w_c[1]:.3f}, {w_c[2]:.3f}] -> {cat_c} ({min_c:.1f} min)"
    )
    print(
        f"  Cuántico: probs=[{w_q[0]:.3f}, {w_q[1]:.3f}, {w_q[2]:.3f}] -> {cat_q} ({min_q:.1f} min)"
    )
    print(
        f"  Categoría coincide: {coinciden_cat}  |  Diferencia = {abs(min_c - min_q):.1f} min\n"
    )

Carga=10, Grasa=10
  Clásico:  pesos=[1.000, 0.000, 0.000] -> Eco (30.0 min)
  Cuántico: probs=[0.333, 0.000, 0.000] -> Eco (30.0 min)
  Categoría coincide: SI  |  Diferencia = 0.0 min

Carga=50, Grasa=20
  Clásico:  pesos=[1.000, 0.000, 0.000] -> Eco (30.0 min)
  Cuántico: probs=[0.333, 0.000, 0.000] -> Eco (30.0 min)
  Categoría coincide: SI  |  Diferencia = 0.0 min

Carga=65, Grasa=40
  Clásico:  pesos=[0.333, 0.500, 0.500] -> Normal (63.8 min)
  Cuántico: probs=[0.056, 0.167, 0.111] -> Normal (65.0 min)
  Categoría coincide: SI  |  Diferencia = 1.3 min

Carga=30, Grasa=70
  Clásico:  pesos=[0.333, 0.667, 0.333] -> Normal (60.0 min)
  Cuántico: probs=[0.074, 0.185, 0.074] -> Normal (60.0 min)
  Categoría coincide: SI  |  Diferencia = 0.0 min

Carga=80, Grasa=80
  Clásico:  pesos=[0.000, 0.000, 1.000] -> Intensivo (90.0 min)
  Cuántico: probs=[0.000, 0.000, 0.333] -> Intensivo (90.0 min)
  Categoría coincide: SI  |  Diferencia = 0.0 min



### Conclusiones

La **categoría de lavado coincide en todos los casos de prueba**, lo que valida que el circuito cuántico implementa correctamente la lógica del sistema.

Los minutos exactos pueden diferir ligeramente en zonas de transición (ej: carga=65, grasa=40 -> 63.8 min clásico vs 65.0 min cuántico) porque:

- El modelo clásico usa `max` y `min` para combinar reglas.
- El modelo cuántico usa probabilidades derivadas de amplitudes.

En los extremos del dominio, donde no hay ambigüedad, los resultados coinciden exactamente.